In [2]:
import pandas as pd
import numpy as np

# ── TASK 1: Clean nav_history.csv ──────────────────
nav = pd.read_csv('02_nav_history.csv')
nav['date'] = pd.to_datetime(nav['date'])
nav = nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)
nav = nav.drop_duplicates(subset=['amfi_code', 'date'])
nav = nav[nav['nav'] > 0]

# Forward-fill for holidays/weekends
all_funds = nav['amfi_code'].unique()
date_min = nav['date'].min()
date_max = nav['date'].max()
full_dates = pd.date_range(start=date_min, end=date_max, freq='D')

filled_parts = []
for fund in all_funds:
    fund_df = nav[nav['amfi_code'] == fund].set_index('date')
    fund_df = fund_df.reindex(full_dates)
    fund_df['amfi_code'] = fund
    fund_df['nav'] = fund_df['nav'].ffill()
    fund_df.index.name = 'date'
    filled_parts.append(fund_df.reset_index())

nav_clean = pd.concat(filled_parts, ignore_index=True)
nav_clean = nav_clean[nav_clean['nav'] > 0].dropna(subset=['nav'])
nav_clean = nav_clean.sort_values(['amfi_code', 'date']).reset_index(drop=True)
print(f"nav_history cleaned: {nav_clean.shape}")

# ── TASK 2: Clean investor_transactions.csv ─────────
txn = pd.read_csv('08_investor_transactions.csv')
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'])
txn['transaction_type'] = txn['transaction_type'].str.strip()
txn.loc[txn['transaction_type'].str.upper() == 'SIP', 'transaction_type'] = 'SIP'
txn = txn[txn['amount_inr'] > 0]
txn['kyc_status'] = txn['kyc_status'].str.strip().str.upper()
txn = txn.drop_duplicates().reset_index(drop=True)
print(f"investor_transactions cleaned: {txn.shape}")

# ── TASK 3: Clean scheme_performance.csv ────────────
perf = pd.read_csv('07_scheme_performance.csv')
for col in ['return_1yr_pct','return_3yr_pct','return_5yr_pct','benchmark_3yr_pct']:
    perf[col] = pd.to_numeric(perf[col], errors='coerce')

def flag_anomaly(row):
    for col in ['return_1yr_pct','return_3yr_pct','return_5yr_pct']:
        if pd.notna(row[col]) and (row[col] < -50 or row[col] > 60):
            return True
    return False

perf['anomaly_flag'] = perf.apply(flag_anomaly, axis=1)
perf['expense_ratio_flag'] = (perf['expense_ratio_pct'] < 0.1) | (perf['expense_ratio_pct'] > 2.5)
perf = perf.drop_duplicates().reset_index(drop=True)
print(f" scheme_performance cleaned: {perf.shape}")

nav_history cleaned: (64320, 3)
investor_transactions cleaned: (32778, 13)
 scheme_performance cleaned: (40, 21)


In [4]:
import sqlite3

# Create database file
conn = sqlite3.connect('bluestock_mf.db')

# Create all tables
conn.executescript("""
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code          INTEGER PRIMARY KEY,
    scheme_name        TEXT NOT NULL,
    fund_house         TEXT NOT NULL,
    category           TEXT NOT NULL,
    plan               TEXT NOT NULL,
    risk_grade         TEXT,
    morningstar_rating INTEGER
);

CREATE TABLE IF NOT EXISTS dim_date (
    date_id     TEXT PRIMARY KEY,
    date        TEXT NOT NULL,
    day         INTEGER NOT NULL,
    month       INTEGER NOT NULL,
    month_name  TEXT NOT NULL,
    quarter     INTEGER NOT NULL,
    year        INTEGER NOT NULL,
    is_weekend  INTEGER NOT NULL,
    day_of_week TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS fact_nav (
    nav_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER NOT NULL,
    date_id   TEXT NOT NULL,
    nav       REAL NOT NULL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date_id)   REFERENCES dim_date(date_id)
);

CREATE TABLE IF NOT EXISTS fact_transactions (
    txn_id             INTEGER PRIMARY KEY AUTOINCREMENT,
    investor_id        TEXT NOT NULL,
    amfi_code          INTEGER NOT NULL,
    date_id            TEXT NOT NULL,
    transaction_type   TEXT NOT NULL,
    amount_inr         REAL NOT NULL,
    state              TEXT,
    city               TEXT,
    city_tier          TEXT,
    age_group          TEXT,
    gender             TEXT,
    annual_income_lakh REAL,
    payment_mode       TEXT,
    kyc_status         TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date_id)   REFERENCES dim_date(date_id)
);

CREATE TABLE IF NOT EXISTS fact_performance (
    perf_id            INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code          INTEGER NOT NULL,
    return_1yr_pct     REAL,
    return_3yr_pct     REAL,
    return_5yr_pct     REAL,
    benchmark_3yr_pct  REAL,
    alpha              REAL,
    beta               REAL,
    sharpe_ratio       REAL,
    sortino_ratio      REAL,
    std_dev_ann_pct    REAL,
    max_drawdown_pct   REAL,
    expense_ratio_pct  REAL,
    anomaly_flag       INTEGER DEFAULT 0,
    expense_ratio_flag INTEGER DEFAULT 0,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_aum (
    aum_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER NOT NULL UNIQUE,
    aum_crore REAL NOT NULL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);
""")
print(" All tables created")

# Load data into tables
dim_fund = perf[['amfi_code','scheme_name','fund_house','category',
                  'plan','risk_grade','morningstar_rating']].drop_duplicates()
dim_fund.to_sql('dim_fund', conn, if_exists='replace', index=False)

all_dates = pd.concat([nav_clean['date'], txn['transaction_date']]).drop_duplicates().dropna().sort_values()
dim_date = pd.DataFrame({'date': all_dates})
dim_date['date_id']     = dim_date['date'].dt.strftime('%Y-%m-%d')
dim_date['day']         = dim_date['date'].dt.day
dim_date['month']       = dim_date['date'].dt.month
dim_date['month_name']  = dim_date['date'].dt.strftime('%B')
dim_date['quarter']     = dim_date['date'].dt.quarter
dim_date['year']        = dim_date['date'].dt.year
dim_date['is_weekend']  = dim_date['date'].dt.dayofweek.isin([5,6]).astype(int)
dim_date['day_of_week'] = dim_date['date'].dt.strftime('%A')
dim_date['date']        = dim_date['date'].dt.strftime('%Y-%m-%d')
dim_date = dim_date.drop_duplicates(subset='date_id')
dim_date.to_sql('dim_date', conn, if_exists='replace', index=False)

nav_clean['date_id'] = nav_clean['date'].dt.strftime('%Y-%m-%d')
nav_clean[['amfi_code','date_id','nav']].to_sql('fact_nav', conn, if_exists='replace', index=False)

txn['date_id'] = txn['transaction_date'].dt.strftime('%Y-%m-%d')
txn[['investor_id','amfi_code','date_id','transaction_type','amount_inr',
     'state','city','city_tier','age_group','gender',
     'annual_income_lakh','payment_mode','kyc_status']].to_sql('fact_transactions', conn, if_exists='replace', index=False)

perf[['amfi_code','return_1yr_pct','return_3yr_pct','return_5yr_pct',
      'benchmark_3yr_pct','alpha','beta','sharpe_ratio','sortino_ratio',
      'std_dev_ann_pct','max_drawdown_pct','expense_ratio_pct',
      'anomaly_flag','expense_ratio_flag']].to_sql('fact_performance', conn, if_exists='replace', index=False)

perf[['amfi_code','aum_crore']].drop_duplicates('amfi_code').to_sql('fact_aum', conn, if_exists='replace', index=False)

conn.commit()

# Verify
print("\n── Row Count Verification ──")
for t in ['dim_fund','dim_date','fact_nav','fact_transactions','fact_performance','fact_aum']:
    c = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:25s}: {c:,} rows")

 All tables created

── Row Count Verification ──
  dim_fund                 : 40 rows
  dim_date                 : 1,608 rows
  fact_nav                 : 64,320 rows
  fact_transactions        : 32,778 rows
  fact_performance         : 40 rows
  fact_aum                 : 40 rows


In [5]:
# Run all 10 analytical queries and display results

queries = {
"Q1 — Top 5 Funds by AUM": """
    SELECT f.scheme_name, f.category, a.aum_crore
    FROM fact_aum a JOIN dim_fund f ON a.amfi_code = f.amfi_code
    ORDER BY a.aum_crore DESC LIMIT 5""",

"Q2 — Avg NAV Per Month": """
    SELECT d.year, d.month_name, ROUND(AVG(n.nav),2) AS avg_nav
    FROM fact_nav n JOIN dim_date d ON n.date_id = d.date_id
    GROUP BY d.year, d.month ORDER BY d.year, d.month""",

"Q3 — SIP YoY Growth": """
    WITH sip AS (
        SELECT d.year, COUNT(*) AS cnt, SUM(t.amount_inr) AS total
        FROM fact_transactions t JOIN dim_date d ON t.date_id = d.date_id
        WHERE t.transaction_type = 'SIP' GROUP BY d.year
    )
    SELECT year, cnt, ROUND(total,2) AS total_amount,
           ROUND((cnt - LAG(cnt) OVER (ORDER BY year)) * 100.0
                 / NULLIF(LAG(cnt) OVER (ORDER BY year),0), 2) AS yoy_growth_pct
    FROM sip ORDER BY year""",

"Q4 — Transactions by State": """
    SELECT state, COUNT(*) AS transactions,
           ROUND(SUM(amount_inr)/1e7,2) AS total_crore
    FROM fact_transactions GROUP BY state ORDER BY transactions DESC""",

"Q5 — Funds with Expense Ratio < 1%": """
    SELECT f.scheme_name, f.plan, p.expense_ratio_pct, p.return_1yr_pct
    FROM fact_performance p JOIN dim_fund f ON p.amfi_code = f.amfi_code
    WHERE p.expense_ratio_pct < 1.0 ORDER BY p.expense_ratio_pct""",

"Q6 — Top 5 Funds by 1yr Return": """
    SELECT f.scheme_name, f.category, p.return_1yr_pct, p.sharpe_ratio
    FROM fact_performance p JOIN dim_fund f ON p.amfi_code = f.amfi_code
    ORDER BY p.return_1yr_pct DESC LIMIT 5""",

"Q7 — Monthly Transaction Volume": """
    SELECT d.year, d.month_name, t.transaction_type, COUNT(*) AS count,
           ROUND(SUM(t.amount_inr)/1e5,2) AS amount_lakh
    FROM fact_transactions t JOIN dim_date d ON t.date_id = d.date_id
    GROUP BY d.year, d.month, t.transaction_type ORDER BY d.year, d.month""",

"Q8 — Fund vs Benchmark": """
    SELECT f.scheme_name, p.return_3yr_pct, p.benchmark_3yr_pct,
           ROUND(p.return_3yr_pct - p.benchmark_3yr_pct,2) AS excess_return,
           CASE WHEN p.return_3yr_pct > p.benchmark_3yr_pct
                THEN 'Outperformer' ELSE 'Underperformer' END AS result
    FROM fact_performance p JOIN dim_fund f ON p.amfi_code = f.amfi_code
    ORDER BY excess_return DESC""",

"Q9 — Transactions by Age Group": """
    SELECT age_group, transaction_type, COUNT(*) AS count,
           ROUND(AVG(amount_inr),2) AS avg_amount
    FROM fact_transactions
    GROUP BY age_group, transaction_type ORDER BY age_group""",

"Q10 — Most Volatile Funds": """
    SELECT f.scheme_name, f.risk_grade, p.std_dev_ann_pct,
           p.max_drawdown_pct, p.sharpe_ratio
    FROM fact_performance p JOIN dim_fund f ON p.amfi_code = f.amfi_code
    ORDER BY p.std_dev_ann_pct DESC LIMIT 10"""
}

for title, sql in queries.items():
    print(f"\n{'='*60}")
    print(f"  {title}")
    print('='*60)
    df = pd.read_sql(sql, conn)
    print(df.to_string(index=False))


  Q1 — Top 5 Funds by AUM
                                          scheme_name        category  aum_crore
Mirae Asset Emerging Bluechip Fund - Regular - Growth Large & Mid Cap      49046
        Kotak Emerging Equity Fund - Regular - Growth         Mid Cap      47469
       Nippon India Small Cap Fund - Regular - Growth       Small Cap      43630
           DSP Top 100 Equity Fund - Regular - Growth       Large Cap      41828
                  UTI Mid Cap Fund - Regular - Growth         Mid Cap      41728

  Q2 — Avg NAV Per Month
 year month_name  avg_nav
 2022    January   207.05
 2022   February   207.72
 2022      March   209.69
 2022      April   211.79
 2022        May   212.73
 2022       June   213.86
 2022       July   213.95
 2022     August   215.70
 2022  September   218.47
 2022    October   219.53
 2022   November   223.51
 2022   December   226.75
 2023    January   230.62
 2023   February   233.83
 2023      March   237.90
 2023      April   240.56
 2023        May   

In [6]:
# Save cleaned CSVs
nav_clean.to_csv('nav_history_clean.csv', index=False)
txn.to_csv('investor_transactions_clean.csv', index=False)
perf.to_csv('scheme_performance_clean.csv', index=False)

conn.close()

# Download all files to your computer
from google.colab import files
files.download('nav_history_clean.csv')
files.download('investor_transactions_clean.csv')
files.download('scheme_performance_clean.csv')
files.download('bluestock_mf.db')

print(" All files downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 All files downloaded!
